# 03 - Model Training: SMS Spam Detection

## Overview
This notebook trains and evaluates five different machine learning models for SMS spam classification.

### Models Implemented:
1. **Logistic Regression** - Simple linear baseline
2. **Naive Bayes** - Probabilistic classifier
3. **Random Forest** - Ensemble learning
4. **Support Vector Machine (SVM)** - Margin-based classifier
5. **LSTM Neural Network** - Deep learning approach

---

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import time
import json
import os

# Scikit-learn imports
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score, roc_curve
)

# TensorFlow imports
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping

# Suppress warnings
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print("Libraries imported successfully!")
print(f"TensorFlow version: {tf.__version__}")

## 2. Load Preprocessed Data

In [ ]:
# Load preprocessed data
df = pd.read_csv('../data/processed/preprocessed_spam.csv')

print(f"Dataset loaded: {len(df)} samples")
print(f"Class distribution:")
print(df['label'].value_counts())
df.head()

In [ ]:
# Prepare features and labels
X = df['processed_text'].values
y = df['label_encoded'].values

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {len(X_train)} samples")
print(f"Test set: {len(X_test)} samples")
print(f"\nTraining class distribution:")
print(f"  Ham: {sum(y_train == 0)}, Spam: {sum(y_train == 1)}")
print(f"\nTest class distribution:")
print(f"  Ham: {sum(y_test == 0)}, Spam: {sum(y_test == 1)}")

## 3. Feature Extraction

### TF-IDF for Traditional ML Models

In [ ]:
# TF-IDF Vectorization for traditional ML models
print("Creating TF-IDF features...")

tfidf_vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))

X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

print(f"TF-IDF Training shape: {X_train_tfidf.shape}")
print(f"TF-IDF Test shape: {X_test_tfidf.shape}")
print(f"Vocabulary size: {len(tfidf_vectorizer.vocabulary_)}")

### Tokenization for LSTM

In [ ]:
# Tokenization for LSTM
print("Creating sequences for LSTM...")

MAX_WORDS = 10000
MAX_LEN = 100

tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token='<OOV>')
tokenizer.fit_on_texts(X_train)

X_train_seq = pad_sequences(tokenizer.texts_to_sequences(X_train), maxlen=MAX_LEN, padding='post')
X_test_seq = pad_sequences(tokenizer.texts_to_sequences(X_test), maxlen=MAX_LEN, padding='post')

vocab_size = min(len(tokenizer.word_index) + 1, MAX_WORDS)

print(f"LSTM Training shape: {X_train_seq.shape}")
print(f"LSTM Test shape: {X_test_seq.shape}")
print(f"Vocabulary size: {vocab_size}")

## 4. Helper Functions

In [ ]:
def evaluate_model(y_true, y_pred, y_pred_proba=None):
    """
    Calculate all evaluation metrics.
    
    Parameters:
        y_true: True labels
        y_pred: Predicted labels
        y_pred_proba: Prediction probabilities (optional)
        
    Returns:
        dict: Dictionary of metrics
    """
    metrics = {
        'accuracy': accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred),
        'recall': recall_score(y_true, y_pred),
        'f1_score': f1_score(y_true, y_pred),
    }
    
    if y_pred_proba is not None:
        metrics['roc_auc'] = roc_auc_score(y_true, y_pred_proba)
    
    return metrics


def plot_confusion_matrix(y_true, y_pred, model_name, save_path=None):
    """
    Plot confusion matrix heatmap.
    """
    cm = confusion_matrix(y_true, y_pred)
    
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Ham', 'Spam'],
                yticklabels=['Ham', 'Spam'])
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.title(f'Confusion Matrix: {model_name}')
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()


def print_results(model_name, metrics, training_time):
    """
    Print formatted results.
    """
    print("\n" + "=" * 60)
    print(f"MODEL: {model_name}")
    print("=" * 60)
    print(f"Training Time: {training_time:.2f} seconds")
    print(f"Accuracy:  {metrics['accuracy']:.4f}")
    print(f"Precision: {metrics['precision']:.4f}")
    print(f"Recall:    {metrics['recall']:.4f}")
    print(f"F1-Score:  {metrics['f1_score']:.4f}")
    if 'roc_auc' in metrics:
        print(f"ROC-AUC:   {metrics['roc_auc']:.4f}")
    print("=" * 60)

## 5. Model Training and Evaluation

### Dictionary to store all results

In [ ]:
# Dictionary to store all results
results = {}

### Model 1: Logistic Regression

**Description:** A linear model that estimates class probabilities using the logistic (sigmoid) function. Simple yet effective baseline for text classification.

**Pros:** Fast training, interpretable, works well with sparse features

**Cons:** Assumes linear decision boundary

In [ ]:
print("\n" + "#" * 60)
print("MODEL 1: LOGISTIC REGRESSION")
print("#" * 60)

# Train
start_time = time.time()
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train_tfidf, y_train)
lr_training_time = time.time() - start_time

# Predict
lr_pred = lr_model.predict(X_test_tfidf)
lr_pred_proba = lr_model.predict_proba(X_test_tfidf)[:, 1]

# Evaluate
lr_metrics = evaluate_model(y_test, lr_pred, lr_pred_proba)
lr_metrics['training_time'] = lr_training_time
lr_metrics['y_pred_proba'] = lr_pred_proba
results['Logistic Regression'] = lr_metrics

# Print results
print_results('Logistic Regression', lr_metrics, lr_training_time)
print("\nClassification Report:")
print(classification_report(y_test, lr_pred, target_names=['Ham', 'Spam']))

# Plot confusion matrix
plot_confusion_matrix(y_test, lr_pred, 'Logistic Regression', 
                      '../results/confusion_matrices/lr_confusion_matrix.png')

### Model 2: Naive Bayes

**Description:** A probabilistic classifier based on Bayes' theorem with strong independence assumptions between features. Particularly effective for text classification.

**Pros:** Very fast, works well with small data, good for high-dimensional sparse data

**Cons:** Assumes feature independence

In [ ]:
print("\n" + "#" * 60)
print("MODEL 2: NAIVE BAYES")
print("#" * 60)

# Train
start_time = time.time()
nb_model = MultinomialNB(alpha=1.0)
nb_model.fit(X_train_tfidf, y_train)
nb_training_time = time.time() - start_time

# Predict
nb_pred = nb_model.predict(X_test_tfidf)
nb_pred_proba = nb_model.predict_proba(X_test_tfidf)[:, 1]

# Evaluate
nb_metrics = evaluate_model(y_test, nb_pred, nb_pred_proba)
nb_metrics['training_time'] = nb_training_time
nb_metrics['y_pred_proba'] = nb_pred_proba
results['Naive Bayes'] = nb_metrics

# Print results
print_results('Naive Bayes', nb_metrics, nb_training_time)
print("\nClassification Report:")
print(classification_report(y_test, nb_pred, target_names=['Ham', 'Spam']))

# Plot confusion matrix
plot_confusion_matrix(y_test, nb_pred, 'Naive Bayes',
                      '../results/confusion_matrices/nb_confusion_matrix.png')

### Model 3: Random Forest

**Description:** An ensemble learning method that constructs multiple decision trees during training and outputs the mode of their predictions.

**Pros:** Handles non-linearity, robust to outliers, provides feature importance

**Cons:** Slower than linear models, can overfit on noisy data

In [ ]:
print("\n" + "#" * 60)
print("MODEL 3: RANDOM FOREST")
print("#" * 60)

# Train
start_time = time.time()
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train_tfidf, y_train)
rf_training_time = time.time() - start_time

# Predict
rf_pred = rf_model.predict(X_test_tfidf)
rf_pred_proba = rf_model.predict_proba(X_test_tfidf)[:, 1]

# Evaluate
rf_metrics = evaluate_model(y_test, rf_pred, rf_pred_proba)
rf_metrics['training_time'] = rf_training_time
rf_metrics['y_pred_proba'] = rf_pred_proba
results['Random Forest'] = rf_metrics

# Print results
print_results('Random Forest', rf_metrics, rf_training_time)
print("\nClassification Report:")
print(classification_report(y_test, rf_pred, target_names=['Ham', 'Spam']))

# Plot confusion matrix
plot_confusion_matrix(y_test, rf_pred, 'Random Forest',
                      '../results/confusion_matrices/rf_confusion_matrix.png')

### Model 4: Support Vector Machine (SVM)

**Description:** Finds the optimal hyperplane that maximizes the margin between classes in high-dimensional space.

**Pros:** Effective in high dimensions, memory efficient, versatile with different kernels

**Cons:** Slow on large datasets, sensitive to feature scaling

In [ ]:
print("\n" + "#" * 60)
print("MODEL 4: SUPPORT VECTOR MACHINE (SVM)")
print("#" * 60)

# Train (using linear kernel for text classification)
start_time = time.time()
svm_model = SVC(kernel='linear', probability=True, random_state=42)
svm_model.fit(X_train_tfidf, y_train)
svm_training_time = time.time() - start_time

# Predict
svm_pred = svm_model.predict(X_test_tfidf)
svm_pred_proba = svm_model.predict_proba(X_test_tfidf)[:, 1]

# Evaluate
svm_metrics = evaluate_model(y_test, svm_pred, svm_pred_proba)
svm_metrics['training_time'] = svm_training_time
svm_metrics['y_pred_proba'] = svm_pred_proba
results['SVM'] = svm_metrics

# Print results
print_results('SVM', svm_metrics, svm_training_time)
print("\nClassification Report:")
print(classification_report(y_test, svm_pred, target_names=['Ham', 'Spam']))

# Plot confusion matrix
plot_confusion_matrix(y_test, svm_pred, 'SVM',
                      '../results/confusion_matrices/svm_confusion_matrix.png')

### Model 5: LSTM Neural Network

**Description:** A type of recurrent neural network (RNN) that can learn long-term dependencies in sequential data. Uses memory cells to capture context in text.

**Pros:** Captures sequential patterns, learns complex relationships, state-of-the-art performance

**Cons:** Requires more data, longer training time, less interpretable

In [ ]:
print("\n" + "#" * 60)
print("MODEL 5: LSTM NEURAL NETWORK")
print("#" * 60)

# Build LSTM model
lstm_model = Sequential([
    # Embedding layer: converts word indices to dense vectors
    Embedding(input_dim=vocab_size, output_dim=128, input_length=MAX_LEN),
    
    # Bidirectional LSTM layer
    Bidirectional(LSTM(64, return_sequences=True)),
    Dropout(0.3),
    
    # Second LSTM layer
    LSTM(32),
    Dropout(0.3),
    
    # Dense layers
    Dense(32, activation='relu'),
    Dropout(0.3),
    
    # Output layer
    Dense(1, activation='sigmoid')
])

# Compile model
lstm_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Model summary
print("\nModel Architecture:")
lstm_model.summary()

In [ ]:
# Early stopping callback
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

# Train LSTM
print("\nTraining LSTM...")
start_time = time.time()

history = lstm_model.fit(
    X_train_seq, y_train,
    epochs=10,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=1
)

lstm_training_time = time.time() - start_time
print(f"\nTraining completed in {lstm_training_time:.2f} seconds")

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy
axes[0].plot(history.history['accuracy'], label='Training')
axes[0].plot(history.history['val_accuracy'], label='Validation')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].set_title('LSTM Training History: Accuracy')
axes[0].legend()
axes[0].grid(True)

# Loss
axes[1].plot(history.history['loss'], label='Training')
axes[1].plot(history.history['val_loss'], label='Validation')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].set_title('LSTM Training History: Loss')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.savefig('../results/plots/lstm_training_history.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Evaluate LSTM
lstm_pred_proba = lstm_model.predict(X_test_seq, verbose=0).flatten()
lstm_pred = (lstm_pred_proba > 0.5).astype(int)

# Calculate metrics
lstm_metrics = evaluate_model(y_test, lstm_pred, lstm_pred_proba)
lstm_metrics['training_time'] = lstm_training_time
lstm_metrics['y_pred_proba'] = lstm_pred_proba
results['LSTM'] = lstm_metrics

# Print results
print_results('LSTM', lstm_metrics, lstm_training_time)
print("\nClassification Report:")
print(classification_report(y_test, lstm_pred, target_names=['Ham', 'Spam']))

# Plot confusion matrix
plot_confusion_matrix(y_test, lstm_pred, 'LSTM',
                      '../results/confusion_matrices/lstm_confusion_matrix.png')

## 6. Results Summary

In [ ]:
# Create comparison table
comparison_data = []
for model_name, metrics in results.items():
    comparison_data.append({
        'Model': model_name,
        'Accuracy': f"{metrics['accuracy']:.4f}",
        'Precision': f"{metrics['precision']:.4f}",
        'Recall': f"{metrics['recall']:.4f}",
        'F1-Score': f"{metrics['f1_score']:.4f}",
        'ROC-AUC': f"{metrics.get('roc_auc', 0):.4f}",
        'Training Time (s)': f"{metrics['training_time']:.2f}"
    })

comparison_df = pd.DataFrame(comparison_data)

# Sort by F1-Score
comparison_df['F1_numeric'] = comparison_df['F1-Score'].astype(float)
comparison_df = comparison_df.sort_values('F1_numeric', ascending=False).drop('F1_numeric', axis=1)

print("\n" + "=" * 80)
print("MODEL COMPARISON SUMMARY")
print("=" * 80)
print(comparison_df.to_string(index=False))
print("=" * 80)

# Save comparison table
comparison_df.to_csv('../results/metrics/model_comparison.csv', index=False)
print("\nComparison table saved to '../results/metrics/model_comparison.csv'")

In [ ]:
# Save detailed results to JSON
results_to_save = {}
for model_name, metrics in results.items():
    results_to_save[model_name] = {
        k: v if not isinstance(v, np.ndarray) else v.tolist()
        for k, v in metrics.items()
        if k != 'y_pred_proba'
    }

with open('../results/metrics/all_results.json', 'w') as f:
    json.dump(results_to_save, f, indent=2)

print("Detailed results saved to '../results/metrics/all_results.json'")

## 7. ROC Curves Comparison

In [ ]:
# Plot ROC curves for all models
plt.figure(figsize=(10, 8))

colors = ['#e74c3c', '#3498db', '#2ecc71', '#9b59b6', '#f39c12']

for (model_name, metrics), color in zip(results.items(), colors):
    if 'y_pred_proba' in metrics:
        fpr, tpr, _ = roc_curve(y_test, metrics['y_pred_proba'])
        auc = metrics['roc_auc']
        plt.plot(fpr, tpr, color=color, linewidth=2, 
                 label=f'{model_name} (AUC = {auc:.4f})')

plt.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random Classifier')

plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curves Comparison: All Models', fontsize=14)
plt.legend(loc='lower right')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../results/plots/roc_curves_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

## 8. Key Findings

### Summary:
All five models have been successfully trained and evaluated on the SMS spam classification task.

In [ ]:
# Find best performing model
best_model = max(results.items(), key=lambda x: x[1]['f1_score'])
fastest_model = min(results.items(), key=lambda x: x[1]['training_time'])

print("\n" + "=" * 60)
print("KEY FINDINGS")
print("=" * 60)
print(f"\nBest Overall Model (by F1-Score): {best_model[0]}")
print(f"  - Accuracy:  {best_model[1]['accuracy']:.4f}")
print(f"  - F1-Score:  {best_model[1]['f1_score']:.4f}")
print(f"  - ROC-AUC:   {best_model[1]['roc_auc']:.4f}")

print(f"\nFastest Model: {fastest_model[0]}")
print(f"  - Training Time: {fastest_model[1]['training_time']:.2f}s")

print("\nRecommendation:")
print(f"  For production use, {best_model[0]} is recommended as it")
print(f"  achieves the best balance between precision and recall.")
print("=" * 60)

---
**Next Step:** Proceed to `04_model_comparison.ipynb` for detailed comparative analysis.